## **Projeto:** Merca Data Platform
### **Squad:** 2 | Diagnostico Bronze — Propensao a Cancelamento
### Objetivo deste notebook
Nao grava nada. Apenas LE a Bronze ja existente (ecommerce_pedidos e ecommerce_itens_pedido) para confirmar, com numeros reais, duas perguntas em aberto antes de escrever a query de feature engineering:
1. Existe duplicidade de id_pedido na Bronze?
2. Qual a distribuicao real de status_pedido (para calcular scale_pos_weight)?

In [0]:
%pip install deltalake

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
from pyspark.sql.functions import col, count as spark_count

df_pedidos = ler_delta("bronze", "ecommerce_pedidos")
df_itens   = ler_delta("bronze", "ecommerce_itens_pedido")

log.info(f"ecommerce_pedidos      : {df_pedidos.count():,} linhas")
log.info(f"ecommerce_itens_pedido : {df_itens.count():,} linhas")

df_pedidos.printSchema()

In [0]:
total_linhas       = df_pedidos.count()
total_ids_distintos = df_pedidos.select("id_pedido").distinct().count()
duplicados          = total_linhas - total_ids_distintos

log.info(f"Total de linhas      : {total_linhas:,}")
log.info(f"id_pedido distintos  : {total_ids_distintos:,}")
log.info(f"Linhas duplicadas    : {duplicados:,}  ({round(100*duplicados/total_linhas, 2)}%)")

if duplicados > 0:
    log.warning("Confirmado: existe duplicidade de id_pedido na Bronze. "
                "A query de feature engineering PRECISA de deduplicacao antes de qualquer agregacao.")
else:
    log.info("Nenhuma duplicidade encontrada. Dedup pode ser dispensada (mas mantemos por seguranca).")

In [0]:
if duplicados > 0:
    df_pedidos.groupBy("id_pedido") \
        .agg(spark_count("*").alias("qtd_ocorrencias")) \
        .filter(col("qtd_ocorrencias") > 1) \
        .orderBy(col("qtd_ocorrencias").desc()) \
        .show(10, truncate=False)

In [0]:
df_status = df_pedidos.groupBy("status_pedido") \
    .agg(spark_count("*").alias("qtd")) \
    .orderBy(col("qtd").desc())

df_status.show(truncate=False)

total = df_pedidos.count()
log.info("Percentuais:")
for row in df_status.collect():
    pct = round(100 * row["qtd"] / total, 2)
    log.info(f"  {row['status_pedido']:<25} {row['qtd']:>8,}  ({pct}%)")

In [0]:
positivos = df_pedidos.filter(col("status_pedido").isin("Cancelado", "ERROR")).count()
negativos = total - positivos

scale_pos_weight = round(negativos / positivos, 4) if positivos > 0 else None

log.info(f"Positivos (Cancelado + ERROR) : {positivos:,}  ({round(100*positivos/total,2)}%)")
log.info(f"Negativos                     : {negativos:,}  ({round(100*negativos/total,2)}%)")
log.info(f"scale_pos_weight calculado    : {scale_pos_weight}")